# 08 - Retrain LightGBM v3

Same recipe as v2 (notebook 05), but also drops `Startup_age` and `founded_year`. Those two are anchored to the 2015 snapshot, so they can't handle 2026 inputs. v2 -> v3 AUC was 0.715 -> 0.707 in the scratch test, a tiny price.

In [1]:
import sys, sklearn, lightgbm, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, precision_recall_curve)
from lightgbm import LGBMClassifier

# note these down, the deployed app must use the same versions
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| joblib", joblib.__version__)

python 3.14.4 | sklearn 1.9.0 | lightgbm 4.7.0 | joblib 1.5.3


In [2]:
df = pd.read_csv("../data/processed/startup_data_final.csv")

DROP = ["label", "funding_duration_days", "Startup_age", "founded_year"]
x = df.drop(columns=DROP)
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42   # same split as before
)
print(x.shape)

(13677, 90)


In [3]:
spw = (y_train == 0).sum() / (y_train == 1).sum()   # < 1 because label 1 = success = majority

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    num_leaves=7,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=5,
    scale_pos_weight=spw,
    random_state=42,
    verbose=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_success = cross_val_predict(lgbm, x_train, y_train, cv=cv, method="predict_proba")[:, 1]

In [4]:
# pick the failure threshold from out-of-fold predictions (train only, no test peeking)
p_fail_oof = 1 - oof_success
prec, rec, thr = precision_recall_curve(1 - y_train, p_fail_oof)
f1 = 2 * prec * rec / (prec + rec + 1e-9)

i = np.argmax(f1[:-1])
best_thr = thr[i]
print("best threshold:", best_thr)
print("failure precision:", prec[i], "| recall:", rec[i], "| F1:", f1[i])

best threshold: 0.5970863625339764
failure precision: 0.2332396627860297 | recall: 0.489469250210615 | F1: 0.31593257161287674


In [5]:
lgbm.fit(x_train, y_train)

p_fail_test = 1 - lgbm.predict_proba(x_test)[:, 1]
y_pred_test = (p_fail_test >= best_thr).astype(int)   # 1 = predicted FAIL

print(classification_report(1 - y_test, y_pred_test, target_names=["success", "fail"]))
print(confusion_matrix(1 - y_test, y_pred_test))
print("ROC-AUC:", roc_auc_score(1 - y_test, p_fail_test))
print("Failure PR-AUC:", average_precision_score(1 - y_test, p_fail_test))

              precision    recall  f1-score   support

     success       0.92      0.81      0.86      2439
        fail       0.22      0.44      0.29       297

    accuracy                           0.77      2736
   macro avg       0.57      0.62      0.58      2736
weighted avg       0.85      0.77      0.80      2736

[[1969  470]
 [ 166  131]]
ROC-AUC: 0.7065944396817706
Failure PR-AUC: 0.24823172892686207


In [6]:
joblib.dump(
    {"model": lgbm, "threshold": best_thr, "columns": list(x_train.columns)},
    "../models/lgbm_model_v3.joblib",
)

# send me this list, I need it for predict.py
print(len(x_train.columns), "columns")
print(list(x_train.columns))

90 columns
['funding_total_usd', 'funding_rounds', 'equity_crowdfunding', 'undisclosed', 'convertible_note', 'debt_financing', 'angel', 'grant', 'product_crowdfunding', 'market_mising', 'category_missing', 'country_missing', 'funding_total_missing', 'days_to_funding', 'funding_per_round', 'single_funding_event', 'funding_stage_reached', 'has_institutional_vc', 'market_Advertising', 'market_Analytics', 'market_Automotive', 'market_Biotechnology', 'market_Clean Technology', 'market_Cloud Computing', 'market_Consulting', 'market_Curated Web', 'market_E-Commerce', 'market_Education', 'market_Enterprise Software', 'market_Entertainment', 'market_Fashion', 'market_Finance', 'market_Games', 'market_Hardware + Software', 'market_Health Care', 'market_Health and Wellness', 'market_Hospitality', 'market_Internet', 'market_Manufacturing', 'market_Media', 'market_Medical', 'market_Messaging', 'market_Mobile', 'market_Music', 'market_Nanotechnology', 'market_Networking', 'market_News', 'market_Phot